In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

In [3]:
movies_df = pd.read_sql("SELECT * FROM movies", engine)
ratings_df = pd.read_sql("SELECT * FROM ratings", engine)
tags_df = pd.read_sql("SELECT * FROM tags", engine)

In [4]:
movies_df.head()

,movieid,title,genres,release_year
0,160838,Into a Dream (2005),drama,2005.0
1,161760,Basta Poco (2015),comedy,2015.0
2,162952,The Ox (1991),drama,1991.0
3,163118,XOXO (2016),drama,2016.0
4,163362,Dixie (1943),comedy,1943.0


In [5]:
print(movies_df['release_year'].isnull().sum())

620


In [6]:
movies_df['release_year'] = (
    movies_df['release_year']
    .fillna(-1)
    .astype(int)
)

In [7]:
movies_df['release_year'] = (
    movies_df['release_year']
    .astype('Int64')
)

In [8]:
print(movies_df['release_year'].isnull().sum())

0


In [9]:
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   movieid       86537 non-null  int64
 1   title         86537 non-null  str  
 2   genres        86537 non-null  str  
 3   release_year  86537 non-null  Int64
dtypes: Int64(1), int64(1), str(2)
memory usage: 5.8 MB


In [10]:
ratings_df.head()

,userid,movieid,rating,timestamp,rating_date
0,330272,379,4.0,973626475,2000-11-08 01:17:55
1,330272,380,4.0,973625866,2000-11-08 01:07:46
2,330272,428,4.0,973624609,2000-11-08 00:46:49
3,330272,431,4.0,973626760,2000-11-08 01:22:40
4,330272,434,3.0,973626011,2000-11-08 01:10:11


In [11]:
ratings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33832162 entries, 0 to 33832161
Data columns (total 5 columns):
 #   Column       Dtype         
---  ------       -----         
 0   userid       int64         
 1   movieid      int64         
 2   rating       float64       
 3   timestamp    int64         
 4   rating_date  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 1.3 GB


In [12]:
if 'timestamp' in ratings_df.columns:
    ratings_df.drop(columns=['timestamp'], inplace=True)

In [13]:
ratings_df['rating_date'] = pd.to_datetime(
    ratings_df['rating_date']
)

In [14]:
ratings_df.drop_duplicates(inplace=True)
ratings_df.duplicated().sum()

np.int64(0)

In [15]:
ratings_df = ratings_df[
    (ratings_df['rating'] >= 0.5) &
    (ratings_df['rating'] <= 5)
]

In [16]:
tags_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2328315 entries, 0 to 2328314
Data columns (total 5 columns):
 #   Column     Dtype         
---  ------     -----         
 0   userid     int64         
 1   movieid    int64         
 2   tag        str           
 3   timestamp  int64         
 4   tag_date   datetime64[us]
dtypes: datetime64[us](1), int64(3), str(1)
memory usage: 113.4 MB


In [17]:
tags_df.head()

,userid,movieid,tag,timestamp,tag_date
0,691,5618,surreal,1677563637,2023-02-28 11:23:57
1,4442,1673,1980s,1438489972,2015-08-02 10:02:52
2,7716,4454,factory,1244205636,2009-06-05 18:10:36
3,8942,2571,complex,1672619975,2023-01-02 06:09:35
4,8942,79357,romance,1641000027,2022-01-01 06:50:27


In [18]:
tags_df['tag'].isnull().sum()

np.int64(0)

In [19]:
tags_df = tags_df.dropna(subset=['tag'])

In [20]:
tags_df['tag_date'] = pd.to_datetime(
    tags_df['tag_date']
)

In [21]:
tags_df.drop(columns=['timestamp'],inplace= True)

In [22]:

tags_df.head(1)

,userid,movieid,tag,tag_date
0,691,5618,surreal,2023-02-28 11:23:57


## SAVE PROCESSED

In [23]:
movies_df.to_csv(
    "../3.outputs/movies_processed.csv",
    index=False
)

ratings_df.to_csv(
    "../3.outputs/ratings_processed_main.csv",
    index=False
)

# Reduce tags file size for deployment
tags_deploy = tags_df.sample(
    n=2200000,
    random_state=42
)

tags_deploy.to_csv(
    "../3.outputs/tags_processed.csv",
    index=False
)